# 04 · Positional Encoding

**Belgedeki bölüm:** Embedding › Hibrit ve Tam LLM Giriş Akışı (pozisyon bilgisi)

- Self-attention kelimelerin sırasını bilmez; bu yüzden her token'ın vektörüne konum bilgisi eklenir.
- **Learned:** öğrenilen tablo, eğitimde güncellenir.
- **Sinüsoidal:** sabit sin / cos formülü, güncellenmez.
- **RoPE:** vektörü sıraya göre döndürür, göreceli mesafeyi taşır; en yaygın yöntem.

In [ ]:
context_length = 32 # gpt-4o 128k, gemma-3-1b 32K

In [ ]:
from usta_tokenizer import UstaTokenizer

tokenizer = UstaTokenizer("tokenizer.json")

In [ ]:
import plotly.graph_objects as go
import plotly.offline

def plot_dots(sentences_data, title, dims=[0, 1, 2]):
  data = [
    go.Scatter3d(
      x=sentence_data["words"][:, dims[0]],
      y=sentence_data["words"][:, dims[1]],
      z=sentence_data["words"][:, dims[2]],
      mode="markers+text",
      marker=dict(
        size=6,
        color=sentence_data["color"],
      ),
      text=sentence_data["labels"],
      hoverinfo="text",
    ) for sentence_data in sentences_data
  ]

  layout = go.Layout(
    scene=dict(
      xaxis_title="Sertlik",
      yaxis_title="Parlaklık",
      zaxis_title="Kırmızılık",
    ),
    title=title,
  )

  fig = go.Figure(data=data, layout=layout)
  plotly.offline.iplot(fig)

In [ ]:
import torch

torch.manual_seed(1)
embeddings = torch.nn.Embedding(num_embeddings=64, embedding_dim=4)

In [ ]:
sentence = "the capital of united states and the capital of france"

In [ ]:
tokens = tokenizer.encode(sentence)
tokens.shape

In [ ]:
meanings = embeddings(tokens)
meanings

In [ ]:
sentences = [
  {
    "words": meanings.detach().numpy(),
    "labels": tokenizer.tokenize(sentence),
    "color": "red",
  },
]
plot_dots(sentences, "Sözlük V1")

Basit Position Encoding

In [ ]:
# sentence context space

torch.manual_seed(1)
pos_embeddings = torch.nn.Embedding(num_embeddings=context_length, embedding_dim=4)

pos_meanings = pos_embeddings(torch.tensor([i for i in range(tokens.shape[0])]))

pos_meanings.shape

In [ ]:
meanings_in_context = meanings + pos_meanings
meanings_in_context.shape

In [ ]:
sentences = [
  {
    "words": meanings_in_context.detach().numpy(),
    "labels": tokenizer.tokenize(sentence),
    "color": "red",
  },
]
plot_dots(sentences, "Sentence Context Space")

Sinüsodial Position Encoding

In [ ]:
import math

def get_position_encoding(context_length, embedding_dim, base=10000, device="cpu"):
  p_embeddings = torch.zeros(context_length, embedding_dim, device=device)
  for pos in range(context_length):
    for i in range(embedding_dim // 2):
      p_embeddings[pos, 2 * i] = math.sin(pos / (base ** (2 * i / embedding_dim)))
      if i + 1 < embedding_dim:
        p_embeddings[pos, 2 * i + 1] = math.cos(pos / (base ** (2 * i / embedding_dim)))
  
  return p_embeddings.unsqueeze(0)
      

In [ ]:
pos_embeddings = get_position_encoding(20, 4)
pos_embeddings.shape

In [ ]:
meanings.shape

In [ ]:
meanings_in_sentence = meanings + pos_embeddings
meanings_in_sentence

In [ ]:
sentences = [
  {
    "words": meanings_in_sentence[0].detach().numpy(),
    "labels": tokenizer.tokenize(sentence),
    "color": "red",
  },
  {
    "words": meanings.detach().numpy(),
    "labels": tokenizer.tokenize(sentence),
    "color": "blue",
  },
]
plot_dots(sentences, "Sentence Context Space V2")

Rotary Position Encoding

In [ ]:
def get_rotary_position_encoding(input: torch.Tensor, base=10000, device="cpu"):
  context_length, dimension = input.shape

  assert dimension % 2 == 0, "dimension must be even"

  half_dimension = dimension // 2

  freqs_indices = torch.arange(0, half_dimension, device=device, dtype=torch.float32)

  freqs = 1.0 / (base ** (freqs_indices / dimension))

  positions = torch.arange(0, context_length, device=device, dtype=torch.float32).unsqueeze(1)

  angles = positions * freqs

  sin_angles = torch.sin(angles)
  cos_angles = torch.cos(angles)

  input_even = input[:, :dimension // 2] # [0, 2, 4, ..]
  input_odd = input[:, dimension // 2:] # [1, 3, 5, ..]

  input_even_rotated = input_even * cos_angles - input_odd * sin_angles
  input_odd_rotated = input_even * sin_angles + input_odd * cos_angles
  
  input_rotated = torch.empty_like(input)

  input_rotated[:, :dimension // 2] = input_even_rotated
  input_rotated[:, dimension // 2:] = input_odd_rotated

  return input_rotated

In [ ]:
torch.manual_seed(1)
random_input = torch.randn(context_length, 4)

pos_rotary_encodings = get_rotary_position_encoding(random_input)
pos_rotary_encodings

In [ ]:
meanings.shape

In [ ]:
meanings_with_pos_encodings = get_rotary_position_encoding(meanings)

In [ ]:
meanings_with_pos_encodings, meanings

In [ ]:
sentences = [
  {
    "words": meanings_with_pos_encodings.detach().numpy(),
    "labels": tokenizer.tokenize(sentence),
    "color": "red",
  },
  {
    "words": meanings.detach().numpy(),
    "labels": tokenizer.tokenize(sentence),
    "color": "blue",
  },
]

plot_dots(sentences, "Sentence Context Space V3 Rotary Encoding")

### Model olarak şimdiye kadar yaptığımızı kullanabiliriz.

In [ ]:
import torch
from usta_embedding import UstaEmbedding
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united states and the capital of france"

tokens = u_tokenizer.encode(prompt)

# UstaEmbedding = nn.Embedding + RoPE pozisyon bilgisi (04_Positional_Encoding, usta_embedding.py)
torch.manual_seed(1)
u_embedding = UstaEmbedding(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_embedding(tokens)
u_embedding

In [ ]:
u_embedding(tokens)